In [1]:
import sys
print(sys.executable)

c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\venv\Scripts\python.exe


In [2]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


# StackUp Engineering Academy — Task 1 Validation

This notebook contains validation and evidence for Task 1 of the
Data Engineering Skills Assessment.

## Tasks covered

- Task 1.1 — Load and transform `projects.csv`
- Task 1.2 — Design the data model / Star Schema
- Task 1.3 — Identify and fix data quality issues in `employees.csv`

All validations are performed against the assessment datasets and
generated outputs.

In [3]:
import pandas as pd
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datasets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Project root: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment
Data directory: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\datasets
Output directory: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\outputs


In [4]:
projects = pd.read_csv(
    OUTPUT_DIR / "results" / "vrinda-daga" / "01_foundations" / "projects_clean.csv"
)

print("Shape:", projects.shape)
print("\nColumns:")
print(projects.columns.tolist())

Shape: (500, 17)

Columns:
['project_id', 'project_name', 'department', 'status', 'start_date', 'end_date', 'budget', 'actual_cost', 'project_manager_id', 'priority', 'region', 'budget_variance', 'is_over_budget', 'duration_days', 'budget_utilisation_pct', 'status_category', 'risk_level']


In [5]:
expected_variance = projects["actual_cost"] - projects["budget"]

variance_mismatches = (
    projects["budget_variance"] != expected_variance
).sum()

print("Budget variance mismatches:", variance_mismatches)

Budget variance mismatches: 0


In [6]:
expected_over_budget = (
    projects["actual_cost"] > projects["budget"]
)

over_budget_mismatches = (
    projects["is_over_budget"] != expected_over_budget
).sum()

print("is_over_budget mismatches:", over_budget_mismatches)

is_over_budget mismatches: 0


In [7]:
valid_dates = (
    projects["start_date"].notna()
    & projects["end_date"].notna()
)

expected_duration = (
    pd.to_datetime(projects.loc[valid_dates, "end_date"])
    - pd.to_datetime(projects.loc[valid_dates, "start_date"])
).dt.days

duration_mismatches = (
    projects.loc[valid_dates, "duration_days"] != expected_duration
).sum()

print("Rows with both dates:", valid_dates.sum())
print("Duration mismatches:", duration_mismatches)

Rows with both dates: 214
Duration mismatches: 0


In [8]:
invalid_duration = (
    ~valid_dates
    & projects["duration_days"].notna()
)

print("Duration values where dates are incomplete:", invalid_duration.sum())

Duration values where dates are incomplete: 0


In [9]:
valid_budget = projects["budget"] != 0

expected_utilisation = (
    projects.loc[valid_budget, "actual_cost"]
    / projects.loc[valid_budget, "budget"]
    * 100
)

utilisation_mismatches = (
    projects.loc[valid_budget, "budget_utilisation_pct"]
    - expected_utilisation
).abs() > 0.000001

print("Non-zero budget rows:", valid_budget.sum())
print("Budget utilisation mismatches:", utilisation_mismatches.sum())

Non-zero budget rows: 468
Budget utilisation mismatches: 0


In [10]:
zero_budget = projects["budget"] == 0

print("Zero-budget rows:", zero_budget.sum())
print(
    "Zero-budget rows with non-null utilisation:",
    projects.loc[zero_budget, "budget_utilisation_pct"].notna().sum()
)

Zero-budget rows: 32
Zero-budget rows with non-null utilisation: 0


In [11]:
status_mapping = {
    "In Progress": "Active",
    "Completed": "Closed",
    "Not Started": "Pending",
    "On Hold": "Pending"
}

expected_category = projects["status"].map(status_mapping)

status_mismatches = (
    projects["status_category"] != expected_category
).sum()

print("Status category mismatches:", status_mismatches)
print("\nStatus values:")
print(projects["status"].value_counts())

print("\nStatus categories:")
print(projects["status_category"].value_counts())

Status category mismatches: 0

Status values:
status
Completed      214
In Progress    189
Not Started     50
On Hold         47
Name: count, dtype: int64

Status categories:
status_category
Closed     214
Active     189
Pending     97
Name: count, dtype: int64


In [12]:
expected_risk = pd.Series("Low", index=projects.index)

medium_condition = (
    projects["priority"].eq("High")
    | projects["budget_utilisation_pct"].gt(90)
)

high_condition = (
    projects["priority"].eq("Critical")
    | projects["is_over_budget"]
)

expected_risk.loc[medium_condition] = "Medium"
expected_risk.loc[high_condition] = "High"

risk_mismatches = (
    projects["risk_level"] != expected_risk
).sum()

print("Risk level mismatches:", risk_mismatches)

print("\nActual risk levels:")
print(projects["risk_level"].value_counts())

print("\nExpected risk levels:")
print(expected_risk.value_counts())

Risk level mismatches: 0

Actual risk levels:
risk_level
High      188
Low       167
Medium    145
Name: count, dtype: int64

Expected risk levels:
High      188
Low       167
Medium    145
Name: count, dtype: int64


## Task 1.1 — Validation Result

Task 1.1 transformations were validated against the generated
`projects_clean.csv` output.

All validation checks passed with zero transformation mismatches.

Key results:
- 500 project records processed.
- 17 output columns generated.
- Budget variance calculation validated.
- Over-budget classification validated.
- Project duration calculation validated for records with both dates.
- Budget utilisation calculation validated.
- Status-to-category mapping validated.
- Risk classification validated.

In [1]:
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "task_1_2_validation.duckdb"

con = duckdb.connect(str(DB_PATH))

print("Database:", DB_PATH)
print("DuckDB version:", duckdb.__version__)
print("Connection established")

Database: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\task_1_2_validation.duckdb
DuckDB version: 1.5.5
Connection established


In [4]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "datasets"

print("Data directory:", DATA_DIR)

history = pd.read_csv(DATA_DIR / "employees_salary_history.csv")

print("Rows:", len(history))
print("Employees with history:", history["employee_id"].nunique())

duplicates = history[
    history.duplicated(
        ["employee_id", "effective_date"],
        keep=False
    )
].sort_values(["employee_id", "effective_date"])

print("\nDuplicate employee/effective_date records:")
display(duplicates)

Data directory: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\datasets
Rows: 1826
Employees with history: 594

Duplicate employee/effective_date records:


,employee_id,previous_salary,new_salary,previous_role,new_role,previous_level,new_level,effective_date,change_type,change_reason
476,EMP0084,15217.0,16314,Vendor Manager,Vendor Manager,Junior,Junior,2025-03-02,Annual Raise,Retention adjustment
477,EMP0084,16314.0,23354,Vendor Manager,Vendor Manager,Junior,Mid,2025-03-02,Promotion,Retention adjustment


In [5]:
same_day_changes = (
    history.groupby(["employee_id", "effective_date"])
    .size()
    .reset_index(name="change_count")
)

same_day_changes = same_day_changes[
    same_day_changes["change_count"] > 1
]

print("Employee/date combinations with multiple changes:",
      len(same_day_changes))

display(same_day_changes)

Employee/date combinations with multiple changes: 1


,employee_id,effective_date,change_count
152,EMP0084,2025-03-02,2


In [7]:
same_day = (
    history[
        history.duplicated(
            ["employee_id", "effective_date"],
            keep=False
        )
    ]
    .sort_values(["employee_id", "effective_date", "new_salary"])
)

display(same_day)

,employee_id,previous_salary,new_salary,previous_role,new_role,previous_level,new_level,effective_date,change_type,change_reason
476,EMP0084,15217.0,16314,Vendor Manager,Vendor Manager,Junior,Junior,2025-03-02,Annual Raise,Retention adjustment
477,EMP0084,16314.0,23354,Vendor Manager,Vendor Manager,Junior,Mid,2025-03-02,Promotion,Retention adjustment


Validation finding: The salary history contains one employee/date combination with multiple changes: EMP0084 on 2025-03-02. Both source records are retained. Since no timestamp is provided, a deterministic secondary ordering is required when constructing SCD Type 2 versions.

## Task 1.2 — Star Schema Validation

The following checks validate that the required warehouse tables were
created successfully in DuckDB and that their structures meet the
assessment requirements.

In [9]:
import duckdb

import duckdb
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment")

DB_PATH = PROJECT_ROOT / "assessment.duckdb"

print("Database:", DB_PATH)
print("Exists:", DB_PATH.exists())

con = duckdb.connect(str(DB_PATH))

tables = [
    "dim_date",
    "dim_project",
    "dim_employee",
    "dim_vendor",
    "bridge_employee_project",
    "fact_transactions"
]

for table in tables:
    print(f"\n--- {table} ---")
    display(con.sql(f"DESCRIBE {table}").df())

Database: C:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\assessment.duckdb
Exists: True

--- dim_date ---


,column_name,column_type,null,key,default,extra
0,date_key,INTEGER,NO,PRI,None,None
1,full_date,DATE,NO,NaN,None,None
2,year,INTEGER,NO,NaN,None,None
3,quarter,INTEGER,NO,NaN,None,None
4,month,INTEGER,NO,NaN,None,None
5,month_name,VARCHAR,NO,NaN,None,None
6,week,INTEGER,NO,NaN,None,None
7,day,INTEGER,NO,NaN,None,None
8,day_of_week,INTEGER,NO,NaN,None,None
9,is_weekend,BOOLEAN,NO,NaN,None,None



--- dim_project ---


,column_name,column_type,null,key,default,extra
0,project_key,INTEGER,NO,PRI,None,None
1,project_id,VARCHAR,NO,UNI,None,None
2,project_name,VARCHAR,YES,NaN,None,None
3,department,VARCHAR,YES,NaN,None,None
4,status,VARCHAR,YES,NaN,None,None
5,start_date,DATE,YES,NaN,None,None
6,end_date,DATE,YES,NaN,None,None
7,budget,"DECIMAL(15,2)",YES,NaN,None,None
8,actual_cost,"DECIMAL(15,2)",YES,NaN,None,None
9,project_manager_id,VARCHAR,YES,NaN,None,None



--- dim_employee ---


...

,column_name,column_type,null,key,default,extra
0,vendor_key,INTEGER,NO,PRI,None,None
1,vendor_id,VARCHAR,NO,UNI,None,None
2,vendor_name,VARCHAR,YES,NaN,None,None



--- bridge_employee_project ---


,column_name,column_type,null,key,default,extra
0,employee_key,INTEGER,NO,PRI,None,None
1,project_key,INTEGER,NO,PRI,None,None



--- fact_transactions ---


,column_name,column_type,null,key,default,extra
0,transaction_key,INTEGER,NO,PRI,None,None
1,transaction_id,VARCHAR,NO,UNI,None,None
2,project_key,INTEGER,NO,NaN,None,None
3,employee_key,INTEGER,YES,NaN,None,None
4,vendor_key,INTEGER,YES,NaN,None,None
5,date_key,INTEGER,NO,NaN,None,None
6,amount,"DECIMAL(15,2)",YES,NaN,None,None
7,category,VARCHAR,YES,NaN,None,None
8,payment_status,VARCHAR,YES,NaN,None,None


In [10]:
print("Tables created:")
display(con.sql("SHOW TABLES").df())

Tables created:


,name
0,bridge_employee_project
1,dim_date
2,dim_employee
3,dim_project
4,dim_vendor
5,fact_transactions


## Task 1.2 — Employee Source Validation

Validate the current employee source before building the SCD Type 2 dimension.

In [11]:
employees = pd.read_csv(DATA_DIR / "employees.csv")

print("Employee rows:", len(employees))
print("Employee columns:")
display(pd.DataFrame({
    "column": employees.columns,
    "dtype": employees.dtypes.astype(str)
}))

Employee rows: 1000
Employee columns:


,column,dtype
employee_id,employee_id,str
full_name,full_name,str
email,email,str
department,department,str
role,role,str
level,level,str
hire_date,hire_date,str
salary,salary,int64
manager_id,manager_id,str
region,region,str


In [12]:
print("Employee source sample:")
display(employees.head())

Employee source sample:


,employee_id,full_name,email,department,role,level,hire_date,salary,manager_id,region,status,years_experience
0,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Mid,2019-08-06,17839,EMP0000,Abu Dhabi,Active,6
1,EMP0002,Mohammed Mansour,mohammed.mansour@presight.ai,Operations,Senior Data Engineer,Senior,2014-08-20,33928,EMP0000,Dubai,Active,11
2,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Mid,2021-02-17,18763,EMP0000,Abu Dhabi,Active,4
3,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Mid,2024-05-30,20100,EMP0000,Dubai,Active,2
4,EMP0005,Priya Chen,priya.chen@presight.ai,Procurement,Procurement Analyst,Junior,2024-02-21,14370,EMP0000,Abu Dhabi,Active,0


In [13]:
sql_path = PROJECT_ROOT / "solutions" / "submissions" / "vrinda-daga" / "01_foundations" / "data_model.sql"

print("SQL file:", sql_path)
print("Exists:", sql_path.exists())

SQL file: C:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\solutions\submissions\vrinda-daga\01_foundations\data_model.sql
Exists: True


In [17]:
with open(sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

con.execute(sql_script)

print("data_model.sql executed successfully")

data_model.sql executed successfully


In [18]:
display(con.sql("SHOW TABLES").df())

,name
0,bridge_employee_project
1,dim_date
2,dim_employee
3,dim_project
4,dim_vendor
5,fact_transactions
6,stg_employees
7,stg_salary_history


In [19]:
print(
    "stg_employees rows:",
    con.sql("SELECT COUNT(*) FROM stg_employees").fetchone()[0]
)

print(
    "stg_salary_history rows:",
    con.sql("SELECT COUNT(*) FROM stg_salary_history").fetchone()[0]
)

stg_employees rows: 1000
stg_salary_history rows: 1826


## SCD Type 2 — Salary History Validation

The employee salary history contains historical salary, role and level
changes for 594 employees.

Before building the SCD Type 2 dimension, we check for multiple changes
for the same employee on the same effective date. This is important
because SCD validity periods must not overlap.

A deterministic ordering will be used for same-day changes.

In [20]:
same_day = con.sql("""
    SELECT
        employee_id,
        effective_date,
        COUNT(*) AS change_count
    FROM stg_salary_history
    GROUP BY employee_id, effective_date
    HAVING COUNT(*) > 1
    ORDER BY employee_id, effective_date
""").df()

print("Employee/date combinations with multiple changes:")
display(same_day)

Employee/date combinations with multiple changes:


,employee_id,effective_date,change_count
0,EMP0084,2025-03-02,2


In [22]:
with open(sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

con.execute(sql_script)

print("data_model.sql executed successfully")

data_model.sql executed successfully


## SCD Type 2 — Dimension Load Validation

Validate that the employee dimension has been populated and that all
1,000 source employees are represented.
Employees with salary history should have multiple dimension versions,
so the total dimension row count is expected to be greater than 1,000.

In [23]:
con.sql("""
    SELECT
        COUNT(*) AS total_dimension_rows,
        COUNT(DISTINCT employee_id) AS unique_employees
    FROM dim_employee
""").df()

,total_dimension_rows,unique_employees
0,2231,1000


In [24]:
current_duplicates = con.sql("""
    SELECT
        employee_id,
        COUNT(*) AS current_count
    FROM dim_employee
    WHERE is_current = TRUE
    GROUP BY employee_id
    HAVING COUNT(*) > 1
    ORDER BY employee_id
""").df()

print("Employees with multiple current records:", len(current_duplicates))

display(current_duplicates)

Employees with multiple current records: 0


,employee_id,current_count


## SCD Type 2 — Q2: Employee Version History

Show employees with multiple dimension versions to confirm that
historical salary, role and level changes are being preserved.

Employees with history should have multiple versions, while employees
without history should normally have a single current version.

In [25]:
version_history = con.sql("""
    SELECT
        employee_id,
        COUNT(*) AS version_count
    FROM dim_employee
    GROUP BY employee_id
    ORDER BY version_count DESC
    LIMIT 10
""").df()

display(version_history)

,employee_id,version_count
0,EMP0166,5
1,EMP0244,5
2,EMP0281,5
3,EMP0517,5
4,EMP0711,5
5,EMP0949,5
6,EMP0234,5
7,EMP0238,5
8,EMP0703,5
9,EMP0942,5


## SCD Type 2 — Q3: Overlapping Period Validation

Check for overlapping validity periods between versions of the same
employee.

A valid SCD Type 2 implementation should have no overlapping periods.

Expected result: zero rows.

In [26]:
overlaps = con.sql("""
    SELECT
        a.employee_id,
        a.employee_key AS version_1,
        a.valid_from AS version_1_from,
        a.valid_to AS version_1_to,
        b.employee_key AS version_2,
        b.valid_from AS version_2_from,
        b.valid_to AS version_2_to
    FROM dim_employee a
    JOIN dim_employee b
        ON a.employee_id = b.employee_id
        AND a.employee_key < b.employee_key
    WHERE a.valid_from < b.valid_to
      AND b.valid_from < a.valid_to
    ORDER BY a.employee_id, a.valid_from
""").df()

print("Overlapping employee periods:", len(overlaps))

display(overlaps)

Overlapping employee periods: 0


,employee_id,version_1,version_1_from,version_1_to,version_2,version_2_from,version_2_to


In [27]:
current_check = con.sql("""
    SELECT
        COUNT(*) AS employees_without_one_current_record
    FROM (
        SELECT
            employee_id,
            COUNT(*) AS current_count
        FROM dim_employee
        WHERE is_current = TRUE
        GROUP BY employee_id
        HAVING COUNT(*) <> 1
    )
""").df()

display(current_check)

,employees_without_one_current_record
0,0


In [28]:
with open(sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

con.execute(sql_script)

print("data_model.sql executed successfully")

data_model.sql executed successfully


In [30]:
con.sql("""
    SELECT
        COUNT(*) AS total_dates,
        MIN(full_date) AS min_date,
        MAX(full_date) AS max_date
    FROM dim_date
""").df()

,total_dates,min_date,max_date
0,4018,2020-01-01,2030-12-31


In [31]:
con.sql("""
    SELECT
        full_date,
        year,
        quarter,
        month,
        month_name,
        week,
        day,
        day_of_week,
        is_weekend
    FROM dim_date
    WHERE full_date IN (
        DATE '2026-01-01',
        DATE '2026-01-03',
        DATE '2026-01-04'
    )
    ORDER BY full_date
""").df()

,full_date,year,quarter,month,month_name,week,day,day_of_week,is_weekend
0,2026-01-01,2026,1,1,January,1,1,4,False
1,2026-01-03,2026,1,1,January,1,3,6,True
2,2026-01-04,2026,1,1,January,1,4,0,True


In [32]:
projects_clean = pd.read_csv(
    PROJECT_ROOT / "outputs" / "results" / "vrinda-daga" /
    "01_foundations" / "projects_clean.csv"
)

print("Rows:", len(projects_clean))
print("Columns:", projects_clean.columns.tolist())

display(projects_clean.head())

Rows: 500
Columns: ['project_id', 'project_name', 'department', 'status', 'start_date', 'end_date', 'budget', 'actual_cost', 'project_manager_id', 'priority', 'region', 'budget_variance', 'is_over_budget', 'duration_days', 'budget_utilisation_pct', 'status_category', 'risk_level']


,project_id,project_name,department,status,start_date,end_date,budget,actual_cost,project_manager_id,priority,region,budget_variance,is_over_budget,duration_days,budget_utilisation_pct,status_category,risk_level
0,PRJ0001,MLOps Platform,Sustainability,Completed,2023-06-13,2023-12-01,1200000.0,907809.0,EMP0126,Medium,Dubai,-292191.0,False,171.0,75.65075,Closed,Low
1,PRJ0002,Tax Automation Platform - Q2,Marketing,In Progress,2025-02-28,NaN,800000.0,795986.0,EMP0045,Medium,Abu Dhabi,-4014.0,False,NaN,99.49825,Active,Medium
2,PRJ0003,ERP Consolidation Phase 2,Legal,Completed,2023-11-21,2024-12-16,80000.0,56981.0,EMP0405,Medium,Dubai,-23019.0,False,391.0,71.22625,Closed,Low
3,PRJ0004,Employee Wellness App - Q1,Marketing,Completed,2024-06-09,2025-08-17,800000.0,707314.0,EMP0270,Medium,Abu Dhabi,-92686.0,False,434.0,88.41425,Closed,Low
4,PRJ0005,Mobile Workforce App - Q4,Finance,In Progress,2023-04-25,NaN,2000000.0,1329426.0,EMP0478,High,Ras Al Khaimah,-670574.0,False,NaN,66.47130,Active,Medium


In [33]:
with open(sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

con.execute(sql_script)

print("data_model.sql executed successfully")

data_model.sql executed successfully


In [34]:
con.sql("""
    SELECT
        COUNT(*) AS project_rows,
        COUNT(DISTINCT project_id) AS unique_projects
    FROM dim_project
""").df()

,project_rows,unique_projects
0,500,500


In [35]:
con.sql("""
    SELECT
        project_id,
        project_name,
        status,
        region,
        budget,
        actual_cost,
        budget_variance,
        is_over_budget,
        duration_days,
        budget_utilisation_pct,
        status_category,
        risk_level
    FROM dim_project
    ORDER BY project_id
    LIMIT 5
""").df()

,project_id,project_name,status,region,budget,actual_cost,budget_variance,is_over_budget,duration_days,budget_utilisation_pct,status_category,risk_level
0,PRJ0001,MLOps Platform,Completed,Dubai,1200000.0,907809.0,-292191.0,False,171,75.65,Closed,Low
1,PRJ0002,Tax Automation Platform - Q2,In Progress,Abu Dhabi,800000.0,795986.0,-4014.0,False,<NA>,99.50,Active,Medium
2,PRJ0003,ERP Consolidation Phase 2,Completed,Dubai,80000.0,56981.0,-23019.0,False,391,71.23,Closed,Low
3,PRJ0004,Employee Wellness App - Q1,Completed,Abu Dhabi,800000.0,707314.0,-92686.0,False,434,88.41,Closed,Low
4,PRJ0005,Mobile Workforce App - Q4,In Progress,Ras Al Khaimah,2000000.0,1329426.0,-670574.0,False,<NA>,66.47,Active,Medium


In [36]:
transactions = pd.read_json(
    DATA_DIR / "transactions.json"
)

print("Rows:", len(transactions))
print("Columns:", transactions.columns.tolist())

display(transactions.head())

Rows: 50000
Columns: ['transaction_id', 'project_id', 'vendor_id', 'vendor_name', 'category', 'amount', 'currency', 'transaction_date', 'approved_by', 'payment_status', 'invoice_ref', 'notes']


,transaction_id,project_id,vendor_id,vendor_name,category,amount,currency,transaction_date,approved_by,payment_status,invoice_ref,notes
0,TXN000001,PRJ0053,VND061,FleetTrack Pro,Research & Development,86860.0,AED,2022-11-16,EMP0169,Paid,INV-2022-0001,Hardware procurement
1,TXN000002,PRJ0166,VND091,DataVault Inc,Hardware,149744.0,AED,2024-06-28,EMP0800,Pending,INV-2024-0002,Hardware procurement
2,TXN000003,PRJ0344,VND082,CX Dynamics,Integration,1279.0,AED,2023-09-04,EMP0638,Paid,INV-2023-0003,Implementation services
3,TXN000004,PRJ0308,VND082,CX Dynamics,Software,57088.0,AED,2022-05-02,EMP0063,Paid,INV-2022-0004,Training session
4,TXN000005,PRJ0071,VND082,CX Dynamics,Software,8853.0,AED,2023-05-10,EMP0850,Paid,INV-2023-0005,Penetration testing


In [37]:
with open(sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

con.execute(sql_script)

print("data_model.sql executed successfully")

data_model.sql executed successfully


In [38]:
con.sql("""
    SELECT
        COUNT(*) AS vendor_rows,
        COUNT(DISTINCT vendor_id) AS unique_vendors
    FROM dim_vendor
""").df()

,vendor_rows,unique_vendors
0,25,25


In [39]:
con.sql("""
    SELECT *
    FROM dim_vendor
    ORDER BY vendor_key
    LIMIT 10
""").df()

,vendor_key,vendor_id,vendor_name
0,1,VND008,DataSys Solutions
1,2,VND012,TechBuild LLC
2,3,VND015,Integra Tech
3,4,VND019,AuditPro
4,5,VND021,CloudOps ME
5,6,VND031,HR Connect
6,7,VND037,SecureNet UAE
7,8,VND044,AI Nexus
8,9,VND052,ProcureEdge
9,10,VND061,FleetTrack Pro


In [40]:
con.sql("""
    SELECT
        COUNT(*) AS relationship_rows,
        COUNT(DISTINCT project_id || '-' || approved_by) AS unique_relationships
    FROM read_json_auto(
        'C:/Users/vrinda.daga/projects/stackup-engineering-academy_assessment/datasets/transactions.json'
    )
    WHERE project_id IS NOT NULL
      AND approved_by IS NOT NULL
""").df()

,relationship_rows,unique_relationships
0,47556,42095


In [41]:
with open(sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

con.execute(sql_script)

print("data_model.sql executed successfully")

data_model.sql executed successfully


In [42]:
con.sql("""
    SELECT
        COUNT(*) AS bridge_rows,
        COUNT(DISTINCT employee_key) AS employees,
        COUNT(DISTINCT project_key) AS projects
    FROM bridge_employee_project
""").df()

,bridge_rows,employees,projects
0,42095,432,435


In [43]:
con.sql("""
    SELECT
        employee_key,
        project_key,
        COUNT(*) AS relationship_count
    FROM bridge_employee_project
    GROUP BY employee_key, project_key
    HAVING COUNT(*) > 1
""").df()

,employee_key,project_key,relationship_count


In [44]:
with open(sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

con.execute(sql_script)

print("data_model.sql executed successfully")

data_model.sql executed successfully


In [45]:
con.sql("""
    SELECT
        COUNT(*) AS transaction_rows,
        COUNT(DISTINCT transaction_id) AS unique_transactions
    FROM fact_transactions
""").df()

,transaction_rows,unique_transactions
0,50000,50000


In [46]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(project_key) AS project_mapped,
        COUNT(employee_key) AS employee_mapped,
        COUNT(vendor_key) AS vendor_mapped,
        COUNT(date_key) AS date_mapped
    FROM fact_transactions
""").df()

,total_rows,project_mapped,employee_mapped,vendor_mapped,date_mapped
0,50000,50000,47556,50000,50000


In [47]:
fk_issues = con.sql("""
    SELECT 'project' AS dimension, COUNT(*) AS orphan_rows
    FROM fact_transactions f
    LEFT JOIN dim_project p
        ON f.project_key = p.project_key
    WHERE p.project_key IS NULL

    UNION ALL

    SELECT 'employee', COUNT(*)
    FROM fact_transactions f
    LEFT JOIN dim_employee e
        ON f.employee_key = e.employee_key
    WHERE f.employee_key IS NOT NULL
      AND e.employee_key IS NULL

    UNION ALL

    SELECT 'vendor', COUNT(*)
    FROM fact_transactions f
    LEFT JOIN dim_vendor v
        ON f.vendor_key = v.vendor_key
    WHERE v.vendor_key IS NULL

    UNION ALL

    SELECT 'date', COUNT(*)
    FROM fact_transactions f
    LEFT JOIN dim_date d
        ON f.date_key = d.date_key
    WHERE d.date_key IS NULL
""").df()

display(fk_issues)

,dimension,orphan_rows
0,project,0
1,employee,0
2,vendor,0
3,date,0


# ============================================================================
# TASK 1.3 — EMPLOYEE DATA QUALITY
# ============================================================================

## Objective

Profile the raw `employees.csv` dataset to identify data-quality issues,
then validate that the issues identified in the source data have been
corrected by `clean_employees()` in `etl_pipeline.py`.

The validation covers:

1. Missing values in key columns
2. Invalid date values
3. Implausible numeric values
4. Logical inconsistencies
5. Employee status conflicts

The notebook provides evidence of the issues before and after cleaning.

## 1.3.1 — Profile Raw Employee Data

Load the raw employee dataset and inspect row counts, columns, missing
values and numeric distributions before applying any cleaning rules.

In [48]:
# Load the raw employee dataset for Task 1.3 profiling.
employees = pd.read_csv(
    DATA_DIR / "employees.csv"
)

print("Employee rows:", len(employees))
print("Employee columns:", len(employees.columns))

print("\nEmployee columns:")
print(employees.columns.tolist())

Employee rows: 1000
Employee columns: 12

Employee columns:
['employee_id', 'full_name', 'email', 'department', 'role', 'level', 'hire_date', 'salary', 'manager_id', 'region', 'status', 'years_experience']


### 1.3.1.1 — Missing Value Profile

Check all employee columns for missing values before cleaning.

In [49]:
# Check completeness of the raw employee dataset.
null_summary = (
    employees.isna()
    .sum()
    .to_frame("null_count")
)

display(null_summary)

,null_count
employee_id,0
full_name,0
email,10
department,0
role,0
level,0
hire_date,0
salary,0
manager_id,0
region,0


### 1.3.1.2 — Numeric Value Profile

Inspect salary and years of experience to identify potentially
implausible numeric values.

In [50]:
# Review the distribution of employee numeric fields.
display(
    employees[
        ["salary", "years_experience"]
    ].describe()
)

,salary,years_experience
count,1000.00000,1000.000000
mean,26461.53100,6.257000
std,11297.04856,4.669985
min,12043.00000,-1.000000
25%,18444.75000,3.000000
50%,22947.00000,5.000000
75%,32264.00000,9.000000
max,76507.00000,25.000000


### 1.3.1.3 — Hire Date Validation

Check whether all hire dates can be interpreted as valid dates.

In [51]:
# Identify hire-date values that cannot be parsed as valid dates.
hire_dates = pd.to_datetime(
    employees["hire_date"],
    errors="coerce"
)

invalid_hire_dates = employees[
    hire_dates.isna()
]

print(
    "Invalid/unparseable hire dates:",
    len(invalid_hire_dates)
)

display(
    invalid_hire_dates[
        ["employee_id", "hire_date", "status"]
    ]
)

Invalid/unparseable hire dates: 8


,employee_id,hire_date,status
37,EMP0038,-999,Active
159,EMP0160,99999-01-01,Active
174,EMP0175,-999,Active
394,EMP0395,99999-01-01,Active
550,EMP0551,99999-01-01,Active
599,EMP0600,-999,Active
854,EMP0855,-999,Active
972,EMP0973,-999,Active


### 1.3.1.4 — Salary and Experience Review

Inspect the lowest and highest salary and experience values to identify
records that may fall outside reasonable business ranges.

In [52]:
# Inspect the lowest salary and experience values.
print("Lowest salary / experience records:")
display(
    employees[
        [
            "employee_id",
            "salary",
            "years_experience",
            "hire_date",
            "status"
        ]
    ]
    .sort_values(
        ["salary", "years_experience"]
    )
    .head(20)
)

print("\nHighest salary / experience records:")
display(
    employees[
        [
            "employee_id",
            "salary",
            "years_experience",
            "hire_date",
            "status"
        ]
    ]
    .sort_values(
        ["salary", "years_experience"],
        ascending=False
    )
    .head(20)
)

Lowest salary / experience records:


,employee_id,salary,years_experience,hire_date,status
371,EMP0372,12043,0,2024-08-29,Active
367,EMP0368,12058,0,2024-01-04,Active
314,EMP0315,12079,1,2024-03-28,Active
346,EMP0347,12091,1,2024-01-22,Active
892,EMP0893,12127,3,2022-04-22,Active
307,EMP0308,12156,1,2024-03-08,Active
997,EMP0998,12164,1,2024-04-21,Active
440,EMP0441,12214,2,2024-04-28,Active
488,EMP0489,12223,2,2024-02-12,Active
671,EMP0672,12228,3,2023-01-13,Active



Highest salary / experience records:


,employee_id,salary,years_experience,hire_date,status
355,EMP0356,76507,-1,2024-09-04,Active
899,EMP0900,71212,3,2022-02-13,Active
210,EMP0211,69078,18,2009-09-20,Inactive
91,EMP0092,67841,22,2004-02-06,Active
168,EMP0169,67180,19,2006-09-20,Active
192,EMP0193,67168,12,2013-01-27,Active
866,EMP0867,66797,2,2024-07-15,Active
888,EMP0889,66312,12,2013-08-11,Active
598,EMP0599,64777,15,2011-08-26,Active
27,EMP0028,64478,20,2007-08-11,Active


### 1.3.1.5 — Employee Status Profile

Review employee status values to identify potential status conflicts
or unexpected values.

In [53]:
# Review the distribution of employee status values.
status_summary = (
    employees["status"]
    .value_counts(dropna=False)
    .to_frame("employee_count")
)

display(status_summary)

,employee_count
status,
Active,960
Inactive,40


## 1.3.1.6 — Logical Consistency Checks

Check whether employee attributes are logically consistent with each other,
particularly hire date, years of experience and employment status.

In [54]:
# Check consistency between hire_date and years_experience.
# Use today's date only for identifying suspicious records;
# the cleaning rule will be based on the source-data logic.

parsed_hire_date = pd.to_datetime(
    employees["hire_date"],
    errors="coerce"
)

reference_date = pd.Timestamp("2025-12-31")

calculated_experience = (
    reference_date.year
    - parsed_hire_date.dt.year
)

experience_check = employees.assign(
    parsed_hire_date=parsed_hire_date,
    calculated_experience=calculated_experience
)

# Identify records where recorded experience is greater than
# the approximate number of years since hire.
experience_inconsistencies = experience_check[
    experience_check["parsed_hire_date"].notna()
    & (
        experience_check["years_experience"]
        > experience_check["calculated_experience"] + 1
    )
]

print(
    "Potential hire-date / experience inconsistencies:",
    len(experience_inconsistencies)
)

display(
    experience_inconsistencies[
        [
            "employee_id",
            "hire_date",
            "years_experience",
            "status"
        ]
    ]
)

Potential hire-date / experience inconsistencies: 241


,employee_id,hire_date,years_experience,status
9,EMP0010,2023-01-06,4,Active
11,EMP0012,2020-09-29,7,Active
13,EMP0014,2024-04-12,3,Active
27,EMP0028,2007-08-11,20,Active
29,EMP0030,2017-07-29,10,Active
...,...,...,...,...
987,EMP0988,2017-05-24,10,Active
990,EMP0991,2019-05-21,8,Active
993,EMP0994,2009-01-12,18,Active
995,EMP0996,2024-09-15,3,Active


### 1.3.1.7 — Employment Status Consistency

Check whether employee status is consistent with the employee's hire date.
For example, an employee with an invalid or future hire date should not
normally be represented as an active employee.

In [55]:
# Identify active employees whose hire date is invalid or in the future.

active_date_conflicts = employees[
    employees["status"].eq("Active")
    & (
        parsed_hire_date.isna()
        | (parsed_hire_date > reference_date)
    )
]

print(
    "Active employees with invalid/future hire dates:",
    len(active_date_conflicts)
)

display(
    active_date_conflicts[
        [
            "employee_id",
            "hire_date",
            "status",
            "years_experience"
        ]
    ]
)

Active employees with invalid/future hire dates: 8


,employee_id,hire_date,status,years_experience
37,EMP0038,-999,Active,7
159,EMP0160,99999-01-01,Active,2
174,EMP0175,-999,Active,15
394,EMP0395,99999-01-01,Active,7
550,EMP0551,99999-01-01,Active,9
599,EMP0600,-999,Active,9
854,EMP0855,-999,Active,10
972,EMP0973,-999,Active,4


### 1.3.1.8 — Employee ID and Manager Consistency

Check employee and manager identifiers for duplicates, self-references,
and managers that do not exist in the employee dataset.

In [56]:
# Check employee ID uniqueness.
duplicate_employee_ids = employees[
    employees["employee_id"].duplicated(keep=False)
]

print(
    "Duplicate employee IDs:",
    len(duplicate_employee_ids)
)

# Check whether an employee is assigned as their own manager.
self_managed = employees[
    employees["employee_id"] == employees["manager_id"]
]

print(
    "Employees who are their own manager:",
    len(self_managed)
)

# Check whether manager IDs exist in the employee dataset.
employee_ids = set(employees["employee_id"])

missing_managers = employees[
    ~employees["manager_id"].isin(employee_ids)
]

print(
    "Employees with manager IDs not found in employee dataset:",
    len(missing_managers)
)

display(
    missing_managers[
        ["employee_id", "manager_id", "status"]
    ].head(20)
)

Duplicate employee IDs: 0
Employees who are their own manager: 1
Employees with manager IDs not found in employee dataset: 5


,employee_id,manager_id,status
0,EMP0001,EMP0000,Active
1,EMP0002,EMP0000,Active
2,EMP0003,EMP0000,Active
3,EMP0004,EMP0000,Active
4,EMP0005,EMP0000,Active


### 1.3.1.9 — Manager Reference Review

Review the manager IDs that are not present as employee IDs.

`EMP0000` may represent a root or system-level manager, so it should not
automatically be treated as an invalid employee reference.

In [57]:
# Review manager IDs that are not present in the employee dataset.
manager_reference_check = (
    employees.loc[
        ~employees["manager_id"].isin(employee_ids),
        ["employee_id", "manager_id", "status"]
    ]
    .sort_values("manager_id")
)

display(manager_reference_check)

,employee_id,manager_id,status
0,EMP0001,EMP0000,Active
1,EMP0002,EMP0000,Active
2,EMP0003,EMP0000,Active
3,EMP0004,EMP0000,Active
4,EMP0005,EMP0000,Active


## 1.3.2 — Data Quality Issues Identified

The profiling results identified the following data-quality issues in the
raw employee dataset:

1. **Missing email values** — 10 employees have missing email addresses.
2. **Invalid hire dates** — 8 employees have invalid hire dates, including
   `-999` and `99999-01-01`.
3. **Negative years of experience** — 1 employee has a negative experience
   value.
4. **Self-referencing manager** — 1 employee is assigned as their own manager.

Manager ID `EMP0000` appears for 5 employees and is retained as a valid
root/system manager reference.

Salary values were reviewed and no confirmed salary-quality issue was
identified.

The corresponding remediation rules are implemented in
`clean_employees()` in `etl_pipeline.py`.

## 1.3.3 — Cleaning Rules

The following data-quality remediation rules will be applied:

- Missing email values will be replaced with a consistent placeholder
  based on the employee ID.
- Invalid hire-date values will be converted to null.
- Negative years of experience will be corrected to zero.
- Self-referencing manager relationships will be removed by setting
  `manager_id` to null.
- `EMP0000` manager references will be retained because they represent
  the root/system manager.
- Salary values will be preserved because no confirmed salary-quality
  issue was identified.

## 1.3.4 — Cleaning Validation

The ETL pipeline has been executed successfully. This section validates that
the identified employee data-quality issues were remediated correctly.

The validation compares the cleaned employee output with the expected
quality rules and confirms that no employee records were lost or duplicated.

In [60]:
# Load the employee output generated by etl_pipeline.py.

employees_clean_path = (
    DATA_DIR.parent
    / "outputs"
    / "results"
    / "vrinda-daga"
    / "01_foundations"
    / "employees_clean.csv"
)

employees_clean = pd.read_csv(employees_clean_path)

print("Cleaned employee rows:", len(employees_clean))
print("Cleaned employee columns:", len(employees_clean.columns))
print("Output file:", employees_clean_path)

Cleaned employee rows: 1000
Cleaned employee columns: 12
Output file: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\outputs\results\vrinda-daga\01_foundations\employees_clean.csv


## 1.3.5 — Validate Missing Email Remediation

The raw dataset contained 10 employees with missing email addresses.

The ETL pipeline replaces each missing email with a deterministic placeholder
based on the employee ID. This validation confirms that no email values remain
null in the cleaned employee output.

In [61]:
# Validate that missing email values were remediated.

missing_emails_after = employees_clean["email"].isna().sum()

print("Missing emails after cleaning:", missing_emails_after)

display(
    employees_clean[
        employees_clean["email"].str.contains(
            "@unknown.presight.ai",
            na=False
        )
    ][
        ["employee_id", "email"]
    ]
)

Missing emails after cleaning: 0


,employee_id,email
357,EMP0358,EMP0358@unknown.presight.ai
372,EMP0373,EMP0373@unknown.presight.ai
460,EMP0461,EMP0461@unknown.presight.ai
535,EMP0536,EMP0536@unknown.presight.ai
646,EMP0647,EMP0647@unknown.presight.ai
657,EMP0658,EMP0658@unknown.presight.ai
703,EMP0704,EMP0704@unknown.presight.ai
790,EMP0791,EMP0791@unknown.presight.ai
849,EMP0850,EMP0850@unknown.presight.ai
851,EMP0852,EMP0852@unknown.presight.ai


## 1.3.6 — Validate Invalid Hire-Date Remediation

The raw dataset contained 8 invalid hire-date values, including placeholder
values such as `-999` and invalid future dates such as `99999-01-01`.

The ETL pipeline converts these invalid values to null rather than inventing
replacement dates. This validation confirms that the invalid values no longer
exist in the cleaned output.

In [62]:
# Validate that invalid hire dates were converted to null.

invalid_hire_dates_after = (
    pd.to_datetime(
        employees_clean["hire_date"],
        errors="coerce"
    ).isna().sum()
)

print(
    "Invalid/unparseable hire dates after cleaning:",
    invalid_hire_dates_after
)

display(
    employees_clean[
        employees_clean["hire_date"].isna()
    ][
        ["employee_id", "hire_date", "status", "years_experience"]
    ]
)

Invalid/unparseable hire dates after cleaning: 8


,employee_id,hire_date,status,years_experience
37,EMP0038,NaN,Active,7
159,EMP0160,NaN,Active,2
174,EMP0175,NaN,Active,15
394,EMP0395,NaN,Active,7
550,EMP0551,NaN,Active,9
599,EMP0600,NaN,Active,9
854,EMP0855,NaN,Active,10
972,EMP0973,NaN,Active,4


## 1.3.7 — Validate Negative Years of Experience Remediation

The raw employee dataset contained negative `years_experience` values, which
are logically invalid.

The ETL pipeline corrects negative values to zero. This validation confirms
that no negative experience values remain in the cleaned employee output.

In [63]:
# Validate that negative years of experience were corrected.

negative_experience_after = (
    pd.to_numeric(
        employees_clean["years_experience"],
        errors="coerce"
    ) < 0
).sum()

zero_experience_after = (
    employees_clean["years_experience"] == 0
).sum()

print(
    "Negative years of experience after cleaning:",
    negative_experience_after
)

print(
    "Employees with zero years of experience:",
    zero_experience_after
)

display(
    employees_clean[
        employees_clean["years_experience"] == 0
    ][
        ["employee_id", "years_experience", "hire_date", "status"]
    ]
)

Negative years of experience after cleaning: 0
Employees with zero years of experience: 61


,employee_id,years_experience,hire_date,status
4,EMP0005,0,2024-02-21,Active
12,EMP0013,0,2024-09-05,Active
17,EMP0018,0,2024-04-08,Active
21,EMP0022,0,2024-04-30,Active
34,EMP0035,0,2024-06-02,Active
...,...,...,...,...
902,EMP0903,0,2024-04-14,Active
917,EMP0918,0,2024-05-03,Active
960,EMP0961,0,2024-05-14,Active
970,EMP0971,0,2024-03-18,Active


## 1.3.8 — Validate Self-Referencing Manager Remediation

The raw employee dataset contained one employee whose `manager_id` was the
same as their own `employee_id`.

The ETL pipeline removes this invalid self-referencing relationship by setting
`manager_id` to null. This validation confirms that no employee manages
themselves in the cleaned output.

In [64]:
# Validate that no employee is their own manager.

self_manager_after = (
    employees_clean["employee_id"]
    == employees_clean["manager_id"]
).sum()

print(
    "Employees who are their own manager after cleaning:",
    self_manager_after
)

display(
    employees_clean[
        employees_clean["manager_id"].isna()
    ][
        ["employee_id", "manager_id", "status"]
    ]
)

Employees who are their own manager after cleaning: 0


,employee_id,manager_id,status
8,EMP0009,NaN,Active


## 1.3.9 — Task 1.3 Validation Summary

The cleaned employee output generated by `etl_pipeline.py` has been validated
against the data-quality issues identified during profiling.

The validation confirms that:
- All 10 missing email values were replaced with deterministic placeholders.
- All 8 invalid hire-date values were converted to null.
- All 5 negative years-of-experience values were corrected to zero.
- The single self-referencing manager relationship was removed.
- The employee row count remained unchanged at 1,000 records.

The cleaned dataset is therefore ready for downstream processing.

In [66]:
# Task 1.3 Final validation summary.

print("Task 1.3 — Final Validation Summary")
print("-" * 45)

print(
    "Employee rows:",
    len(employees_clean)
)

print(
    "Missing emails remaining:",
    employees_clean["email"].isna().sum()
)

print(
    "Invalid hire dates converted to null:",
    employees_clean["hire_date"].isna().sum()
)

print(
    "Negative experience values remaining:",
    (
        pd.to_numeric(
            employees_clean["years_experience"],
            errors="coerce"
        ) < 0
    ).sum()
)

print(
    "Self-referencing managers remaining:",
    (
        employees_clean["employee_id"]
        == employees_clean["manager_id"]
    ).sum()
)

Task 1.3 — Final Validation Summary
---------------------------------------------
Employee rows: 1000
Missing emails remaining: 0
Invalid hire dates converted to null: 8
Negative experience values remaining: 0
Self-referencing managers remaining: 0
